In [100]:
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.api import anova_lm

df = pd.read_csv('Concrete_Data.csv')



In [101]:
df.shape


(1030, 9)

In [102]:
df.head()


,Cement (component 1)(kg in a m^3 mixture),Blast Furnace Slag (component 2)(kg in a m^3 mixture),Fly Ash (component 3)(kg in a m^3 mixture),Water (component 4)(kg in a m^3 mixture),Superplasticizer (component 5)(kg in a m^3 mixture),Coarse Aggregate (component 6)(kg in a m^3 mixture),Fine Aggregate (component 7)(kg in a m^3 mixture),Age (day),"Concrete compressive strength(MPa, megapascals)"
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [103]:
df.columns.tolist()

['Cement (component 1)(kg in a m^3 mixture)',
 'Blast Furnace Slag (component 2)(kg in a m^3 mixture)',
 'Fly Ash (component 3)(kg in a m^3 mixture)',
 'Water  (component 4)(kg in a m^3 mixture)',
 'Superplasticizer (component 5)(kg in a m^3 mixture)',
 'Coarse Aggregate  (component 6)(kg in a m^3 mixture)',
 'Fine Aggregate (component 7)(kg in a m^3 mixture)',
 'Age (day)',
 'Concrete compressive strength(MPa, megapascals) ']

In [2]:
!pip install statsmodels

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 22.3 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.9/232.9 KB 4.3 MB/s eta 0:00:00a 0:00:01


In [104]:
y_col = 'Concrete compressive strength(MPa, megapascals) _raw'
X_cols = [
    'Cement (component 1)(kg in a m^3 mixture)_std',
    'Blast Furnace Slag (component 2)(kg in a m^3 mixture)_std',
    'Fly Ash (component 3)(kg in a m^3 mixture)_std',
    'Water  (component 4)(kg in a m^3 mixture)_std',
    'Superplasticizer (component 5)(kg in a m^3 mixture)_std',
    'Coarse Aggregate  (component 6)(kg in a m^3 mixture)_std',
    'Fine Aggregate (component 7)(kg in a m^3 mixture)_std',
    'Age (day)_std'
]

In [105]:
df_std = (df - df.mean()) / df.std()
df = df.add_suffix('_raw').join(df_std.add_suffix('_std'))

In [106]:
df.columns

Index(['Cement (component 1)(kg in a m^3 mixture)_raw',
       'Blast Furnace Slag (component 2)(kg in a m^3 mixture)_raw',
       'Fly Ash (component 3)(kg in a m^3 mixture)_raw',
       'Water  (component 4)(kg in a m^3 mixture)_raw',
       'Superplasticizer (component 5)(kg in a m^3 mixture)_raw',
       'Coarse Aggregate  (component 6)(kg in a m^3 mixture)_raw',
       'Fine Aggregate (component 7)(kg in a m^3 mixture)_raw',
       'Age (day)_raw', 'Concrete compressive strength(MPa, megapascals) _raw',
       'Cement (component 1)(kg in a m^3 mixture)_std',
       'Blast Furnace Slag (component 2)(kg in a m^3 mixture)_std',
       'Fly Ash (component 3)(kg in a m^3 mixture)_std',
       'Water  (component 4)(kg in a m^3 mixture)_std',
       'Superplasticizer (component 5)(kg in a m^3 mixture)_std',
       'Coarse Aggregate  (component 6)(kg in a m^3 mixture)_std',
       'Fine Aggregate (component 7)(kg in a m^3 mixture)_std',
       'Age (day)_std',
       'Concrete compressive

In [107]:
test_df = df.iloc[500:630]                     # rows 500 to 629 since i start at 0
train_df = pd.concat([df.iloc[:500], df.iloc[630:]], ignore_index=True)

In [113]:
import numpy as np
import statsmodels.api as sm




y_col = 'Concrete compressive strength(MPa, megapascals) _raw'
X_cols = [
    'Cement (component 1)(kg in a m^3 mixture)_std',
    'Blast Furnace Slag (component 2)(kg in a m^3 mixture)_std',
    'Fly Ash (component 3)(kg in a m^3 mixture)_std',
    'Water  (component 4)(kg in a m^3 mixture)_std',
    'Superplasticizer (component 5)(kg in a m^3 mixture)_std',
    'Coarse Aggregate  (component 6)(kg in a m^3 mixture)_std',
    'Fine Aggregate (component 7)(kg in a m^3 mixture)_std',
    'Age (day)_std'
]



X_train = train_df[X_cols]
y_train = train_df[y_col]

X_train = sm.add_constant(X_train)


model = sm.OLS(y_train, X_train)
results = model.fit()


y_pred_train = results.predict(X_train)

# Training MSE and RMSE
train_mse = ((y_train - y_pred_train) ** 2).mean()
#train_rmse = train_mse ** 0.5


ss_res_train = ((y_train - y_pred_train) ** 2).sum()
ss_tot_train = ((y_train - y_train.mean()) ** 2).sum()
train_r2 = 1 - (ss_res_train / ss_tot_train)

print(f"\nTRAIN SET PERFORMANCE")
print(f"Train MSE: {train_mse:.4f}")
print(f"Train R-squared (VE): {train_r2:.6f}")


#test
X_test = test_df[X_cols]
y_test = test_df[y_col]

X_test = sm.add_constant(X_test, has_constant='add')


y_pred_test = results.predict(X_test)

# Test MSE and RMSE
test_mse = ((y_test - y_pred_test) ** 2).mean()
#test_rmse = test_mse ** 0.5

ss_res_test = ((y_test - y_pred_test) ** 2).sum()
ss_tot_test = ((y_test - y_test.mean()) ** 2).sum()
test_r2 = 1 - (ss_res_test / ss_tot_test)

print(f"\nTEST SET PERFORMANCE")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test R-squared (VE): {test_r2:.6f}")



TRAIN SET PERFORMANCE
Train MSE: 104.1365
Train R-squared (VE): 0.624082

TEST SET PERFORMANCE
Test MSE: 141.2356
Test R-squared (VE): 0.368810


In [114]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                             OLS Regression Results                                             
================================================================================================================
Dep. Variable:     Concrete compressive strength(MPa, megapascals) _raw   R-squared:                       0.624
Model:                                                              OLS   Adj. R-squared:                  0.621
Method:                                                   Least Squares   F-statistic:                     184.9
Date:                                                  Fri, 17 Oct 2025   Prob (F-statistic):          1.84e-183
Time:                                                          22:13:16   Log-Likelihood:                -3367.6
No. Observations:                                                   900   AIC:                             6753.
Df Residuals:                                                       891   BIC:                             6796.
Df Model:                                                             8                                         
Covariance Type:                                              nonrobust                                         
=============================================================================================================================
                                                                coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------
const                                                        36.5300      0.346    105.645      0.000      35.851      37.209
Cement (component 1)(kg in a m^3 mixture)_std                13.8427      0.990     13.987      0.000      11.900      15.785
Blast Furnace Slag (component 2)(kg in a m^3 mixture)_std    10.7627      0.990     10.868      0.000       8.819      12.706
Fly Ash (component 3)(kg in a m^3 mixture)_std                6.7921      0.919      7.393      0.000       4.989       8.595
Water  (component 4)(kg in a m^3 mixture)_std                -2.8554      0.914     -3.124      0.002      -4.649      -1.061
Superplasticizer (component 5)(kg in a m^3 mixture)_std       0.7104      0.599      1.185      0.236      -0.466       1.887
Coarse Aggregate  (component 6)(kg in a m^3 mixture)_std      2.2245      0.799      2.784      0.005       0.657       3.792
Fine Aggregate (component 7)(kg in a m^3 mixture)_std         2.7155      0.911      2.982      0.003       0.928       4.503
Age (day)_std                                                 7.5049      0.375     20.000      0.000       6.768       8.241
==============================================================================
Omnibus:                       11.371   Durbin-Watson:                   1.244
Prob(Omnibus):                  0.003   Jarque-Bera (JB):               11.564
Skew:                          -0.249   Prob(JB):                      0.00308
Kurtosis:                       3.247   Cond. No.                         9.40
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [115]:

y_col = 'Concrete compressive strength(MPa, megapascals) _raw'
X_cols = [
    'Cement (component 1)(kg in a m^3 mixture)_raw',
    'Blast Furnace Slag (component 2)(kg in a m^3 mixture)_raw',
    'Fly Ash (component 3)(kg in a m^3 mixture)_raw',
    'Water  (component 4)(kg in a m^3 mixture)_raw',
    'Superplasticizer (component 5)(kg in a m^3 mixture)_raw',
    'Coarse Aggregate  (component 6)(kg in a m^3 mixture)_raw',
    'Fine Aggregate (component 7)(kg in a m^3 mixture)_raw',
    'Age (day)_raw'
]



X_train = train_df[X_cols]
y_train = train_df[y_col]

X_train = sm.add_constant(X_train)


model = sm.OLS(y_train, X_train)
results = model.fit()


y_pred_train = results.predict(X_train)

# Training MSE and RMSE
train_mse = ((y_train - y_pred_train) ** 2).mean()
#train_rmse = train_mse ** 0.5


ss_res_train = ((y_train - y_pred_train) ** 2).sum()
ss_tot_train = ((y_train - y_train.mean()) ** 2).sum()
train_r2 = 1 - (ss_res_train / ss_tot_train)

print(f"\nTRAIN SET PERFORMANCE")
print(f"Train MSE: {train_mse:.4f}")
print(f"Train R-squared (VE): {train_r2:.6f}")


#test
X_test = test_df[X_cols]
y_test = test_df[y_col]

X_test = sm.add_constant(X_test, has_constant='add')


y_pred_test = results.predict(X_test)

# Test MSE and RMSE
test_mse = ((y_test - y_pred_test) ** 2).mean()
#test_rmse = test_mse ** 0.5

ss_res_test = ((y_test - y_pred_test) ** 2).sum()
ss_tot_test = ((y_test - y_test.mean()) ** 2).sum()
test_r2 = 1 - (ss_res_test / ss_tot_test)

print(f"\nTEST SET PERFORMANCE")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test R-squared (VE): {test_r2:.6f}")


TRAIN SET PERFORMANCE
Train MSE: 104.1365
Train R-squared (VE): 0.624082

TEST SET PERFORMANCE
Test MSE: 141.2356
Test R-squared (VE): 0.368810


In [116]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                             OLS Regression Results                                             
================================================================================================================
Dep. Variable:     Concrete compressive strength(MPa, megapascals) _raw   R-squared:                       0.624
Model:                                                              OLS   Adj. R-squared:                  0.621
Method:                                                   Least Squares   F-statistic:                     184.9
Date:                                                  Fri, 17 Oct 2025   Prob (F-statistic):          1.84e-183
Time:                                                          22:13:19   Log-Likelihood:                -3367.6
No. Observations:                                                   900   AIC:                             6753.
Df Residuals:                                                       891   BIC:                             6796.
Df Model:                                                             8                                         
Covariance Type:                                              nonrobust                                         
=============================================================================================================================
                                                                coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------
const                                                       -51.6017     28.786     -1.793      0.073    -108.099       4.896
Cement (component 1)(kg in a m^3 mixture)_raw                 0.1325      0.009     13.987      0.000       0.114       0.151
Blast Furnace Slag (component 2)(kg in a m^3 mixture)_raw     0.1247      0.011     10.868      0.000       0.102       0.147
Fly Ash (component 3)(kg in a m^3 mixture)_raw                0.1061      0.014      7.393      0.000       0.078       0.134
Water  (component 4)(kg in a m^3 mixture)_raw                -0.1337      0.043     -3.124      0.002      -0.218      -0.050
Superplasticizer (component 5)(kg in a m^3 mixture)_raw       0.1189      0.100      1.185      0.236      -0.078       0.316
Coarse Aggregate  (component 6)(kg in a m^3 mixture)_raw      0.0286      0.010      2.784      0.005       0.008       0.049
Fine Aggregate (component 7)(kg in a m^3 mixture)_raw         0.0339      0.011      2.982      0.003       0.012       0.056
Age (day)_raw                                                 0.1188      0.006     20.000      0.000       0.107       0.130
==============================================================================
Omnibus:                       11.371   Durbin-Watson:                   1.244
Prob(Omnibus):                  0.003   Jarque-Bera (JB):               11.564
Skew:                          -0.249   Prob(JB):                      0.00308
Kurtosis:                       3.247   Cond. No.                     1.09e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.09e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [119]:
y_col = 'Concrete compressive strength(MPa, megapascals) _raw'
X_cols = [
    'Cement (component 1)(kg in a m^3 mixture)_raw',
    'Blast Furnace Slag (component 2)(kg in a m^3 mixture)_raw',
    'Fly Ash (component 3)(kg in a m^3 mixture)_raw',
    'Water  (component 4)(kg in a m^3 mixture)_raw',
    'Superplasticizer (component 5)(kg in a m^3 mixture)_raw',
    'Coarse Aggregate  (component 6)(kg in a m^3 mixture)_raw',
    'Fine Aggregate (component 7)(kg in a m^3 mixture)_raw',
    'Age (day)_raw'
]

X_train = np.log1p(train_df[X_cols]) #add 1 to everything before tkaing the log
y_train = train_df[y_col]

X_train = sm.add_constant(X_train)

model = sm.OLS(y_train, X_train)
results = model.fit()

y_pred_train = results.predict(X_train)

train_mse = ((y_train - y_pred_train) ** 2).mean()

ss_res_train = ((y_train - y_pred_train) ** 2).sum()
ss_tot_train = ((y_train - y_train.mean()) ** 2).sum()
train_r2 = 1 - (ss_res_train / ss_tot_train)

print(f"\nTRAIN SET PERFORMANCE")
print(f"Train MSE: {train_mse:.4f}")
print(f"Train R-squared (VE): {train_r2:.6f}")

X_test = np.log1p(test_df[X_cols])
y_test = test_df[y_col]

X_test = sm.add_constant(X_test, has_constant='add')

y_pred_test = results.predict(X_test)

test_mse = ((y_test - y_pred_test) ** 2).mean()

ss_res_test = ((y_test - y_pred_test) ** 2).sum()
ss_tot_test = ((y_test - y_test.mean()) ** 2).sum()
test_r2 = 1 - (ss_res_test / ss_tot_test)

print(f"\nTEST SET PERFORMANCE")
print(f"Test MSE: {test_mse:.4f}")
print(f"Test R-squared (VE): {test_r2:.6f}")



TRAIN SET PERFORMANCE
Train MSE: 55.8300
Train R-squared (VE): 0.798462

TEST SET PERFORMANCE
Test MSE: 54.3614
Test R-squared (VE): 0.757056


In [120]:
results.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                             OLS Regression Results                                             
================================================================================================================
Dep. Variable:     Concrete compressive strength(MPa, megapascals) _raw   R-squared:                       0.798
Model:                                                              OLS   Adj. R-squared:                  0.797
Method:                                                   Least Squares   F-statistic:                     441.2
Date:                                                  Fri, 17 Oct 2025   Prob (F-statistic):          9.40e-304
Time:                                                          22:26:48   Log-Likelihood:                -3087.1
No. Observations:                                                   900   AIC:                             6192.
Df Residuals:                                                       891   BIC:                             6235.
Df Model:                                                             8                                         
Covariance Type:                                              nonrobust                                         
=============================================================================================================================
                                                                coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------------------------
const                                                        81.1116     75.959      1.068      0.286     -67.967     230.190
Cement (component 1)(kg in a m^3 mixture)_raw                23.6119      1.076     21.946      0.000      21.500      25.723
Blast Furnace Slag (component 2)(kg in a m^3 mixture)_raw     2.3025      0.176     13.074      0.000       1.957       2.648
Fly Ash (component 3)(kg in a m^3 mixture)_raw                0.1955      0.186      1.048      0.295      -0.171       0.561
Water  (component 4)(kg in a m^3 mixture)_raw               -38.3322      4.030     -9.513      0.000     -46.241     -30.423
Superplasticizer (component 5)(kg in a m^3 mixture)_raw       1.8771      0.418      4.489      0.000       1.056       2.698
Coarse Aggregate  (component 6)(kg in a m^3 mixture)_raw      4.1888      5.032      0.832      0.405      -5.688      14.066
Fine Aggregate (component 7)(kg in a m^3 mixture)_raw        -6.6554      3.906     -1.704      0.089     -14.321       1.011
Age (day)_raw                                                 9.0345      0.234     38.618      0.000       8.575       9.494
==============================================================================
Omnibus:                       16.466   Durbin-Watson:                   1.278
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               26.465
Skew:                           0.129   Prob(JB):                     1.79e-06
Kurtosis:                       3.800   Cond. No.                     4.04e+03
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 4.04e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [86]:
results.mse_resid

np.float64(105.18834537137059)

In [73]:
# results.mse_resid

np.float64(80.08423738041401)